In [12]:
from cellpose import models, io
import pyclesperanto as cle
from bioio import BioImage
import napari
from skimage import exposure
from skimage.segmentation import relabel_sequential
from skimage.transform import rescale, resize
from skimage.io import imsave, imread
from skimage.measure import label, regionprops, regionprops_table, marching_cubes, mesh_surface_area
from scipy import ndimage
import numpy as np
import random
from glob import glob
import pandas as pd
import os
from tqdm import tqdm
from joblib import Parallel, delayed

cle.select_device("NVIDIA")

(OpenCL) NVIDIA RTX 4000 Ada Generation (OpenCL 3.0 CUDA)
	Vendor:                      NVIDIA Corporation
	Driver Version:              582.16
	Device Type:                 GPU
	Compute Units:               48
	Global Memory Size:          20474 MB
	Local Memory Size:           0 MB
	Maximum Buffer Size:         5118 MB
	Max Clock Frequency:         2175 MHz
	Image Support:               Yes

In [ ]:
def read_imgs(path):
    files = sorted(glob(f"{path}/*.czi"))
    names = list(map(os.path.basename,files))
    names = [name[0:-4] for name in names]
    img_data = [BioImage(f).get_image_data("ZYX", C=1) for f in files]
    pixel_spacing = get_pixel_spacing(BioImage(files[0]))
    return img_data, pixel_spacing

def get_pixel_spacing(img):
    ''' 
    Use one image from dataset read in with BioImage without .get_image_data to access pixel spacing
    '''
    if img.dims['Z'][0] > 1:
        pixel_spacing = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])
    else:
        pixel_spacing = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1])
    return pixel_spacing

def normalize_images(input_image,tophat_radius=10):
    input_gpu = cle.push(input_image)
    #normalizing the image stack
    equalized_intensities_stack = cle.create_like(input_gpu)
    a_slice = cle.create([input_gpu.shape[1], input_gpu.shape[2]])
    num_slices = input_gpu.shape[0]
    mean_intensity_stack = cle.mean_of_all_pixels(input_gpu)
    corrected_slice = None
    for z in range(0, num_slices):
        # get a single slice out of the stack
        cle.copy_slice(input_gpu, a_slice, z)
        # measure its intensity
        mean_intensity_slice = cle.mean_of_all_pixels(a_slice)
        # correct the intensity
        correction_factor = mean_intensity_slice/mean_intensity_stack
        corrected_slice = cle.multiply_image_and_scalar(a_slice, corrected_slice, correction_factor)
        # copy slice back in a stack
        cle.copy_slice(corrected_slice, equalized_intensities_stack, z)
    #background subtraction (increase the signal to noise ratio for improved segmentation results)
    background_subtracted_top_hat = cle.top_hat_sphere(equalized_intensities_stack,radius_x=tophat_radius,radius_y=tophat_radius,radius_z=tophat_radius)
    #pull data off gpu
    input_pull = cle.pull(input_gpu)
    background_subtracted_top_hat_pull = cle.pull(background_subtracted_top_hat)
    equalized_intensities_stack_pull = cle.pull(equalized_intensities_stack)
    return background_subtracted_top_hat_pull

def down_size(img, scale=None):
   scale == 1 if scale == None else scale == scale
   if scale == None:
       print('no scaling provided, using scaling of 1 (no scaling applied)')
   return rescale(img, scale, anti_aliasing=True, channel_axis=None,preserve_range=True)

def run_cpsam(imgs):
    '''
    Give function list of all resized images for more efficient processing
    '''
    #io.logger_setup()
    model = models.CellposeModel(gpu=True)
    masks, _, _ = model.eval(imgs, batch_size=32, z_axis=0, do_3D=False,normalize=True,stitch_threshold=0.6,cellprob_threshold=-0.5)
    return masks

def up_size_masks(masks, img):
    if masks.shape == img.shape:
        print('masks and image are the same size, no resizing applied')
        return masks
    else:
        return resize(masks,img.shape,preserve_range=True,order=0).astype(np.uint16)

def clear_xy_borders(mask_img,border):
    bool_arr = np.zeros((mask_img.shape[0],mask_img.shape[1]-border*2,mask_img.shape[2]-border*2))
    xy_boarder = np.pad(bool_arr,pad_width=((0,0),(border,border),(border,border)),mode='constant',constant_values=1).astype(np.bool)
    labels, number = label(mask_img,background=0,return_num=True)
    indices = np.arange(number+1)
    border_indices = np.unique(labels[xy_boarder])
    label_mask = np.isin(indices, border_indices)
    mask = label_mask[labels.reshape(-1)].reshape(labels.shape)
    labels[mask] = 0
    seq_lables, _, _ = relabel_sequential(labels)
    return seq_lables

def _label_sa_solidity_pre(label_val, submask, pixel_spacing):
    m = submask.astype(np.uint8)

    # surface area
    padded = np.pad(m, 1, mode='constant')
    verts, faces, _, _ = marching_cubes(padded, level=0.5, spacing=pixel_spacing)
    surface_area = mesh_surface_area(verts, faces)

    # solidity on the same small crop — convex hull is cheap here
    sub_props = regionprops(m)[0]
    solidity = sub_props.solidity

    return label_val, surface_area, solidity


def measure_objects(mask_img, pixel_spacing, n_jobs=-1):
    # 1. Standard descriptors — one fast pass.
    #    Pass spacing so 'area' is volume in physical units (e.g. µm³).
    print('collecting regions properties')
    props_df = pd.DataFrame(regionprops_table(
        mask_img,
        spacing=pixel_spacing,
        properties=[
            'label', 'area', 'centroid', 'equivalent_diameter_area', 
            'extent', 'axis_major_length', 'axis_minor_length'
        ],
    ))

    # 2. Pre-crop each object in the main process so workers receive only
    #    a tiny bool array, not the full label volume.
    print('finding objects with ndimage')
    objects = ndimage.find_objects(mask_img)
    print(f'extracting subregions for {len(objects)}')
    tasks = []
    for idx, sl in enumerate(objects):
        if sl is None:
            continue
        label_val = idx + 1
        padded_sl = tuple(slice(max(0, s.start - 1), s.stop + 1) for s in sl)
        submask = (mask_img[padded_sl] == label_val)   # small bool array
        tasks.append((label_val, submask))

    # 3. Parallelize only the costly surface-area extraction.
    print('Extracting suface areas for objects')
    results = Parallel(n_jobs=n_jobs)(
        delayed(_label_sa_solidity_pre)(label_val, submask, pixel_spacing)
        for label_val, submask in tqdm(tasks)
    )
    sa_df = pd.DataFrame(results, columns=['label', 'surface_area', 'solidity'])
    # 4. Merge on label — order-independent, gap-safe.
    #print(props_df.columns)
    df = props_df.merge(sa_df, on='label', how='left')
    # 5. Sphericity as a vectorized op, now that V and A are aligned columns.
    #    'area' is volume because spacing was passed to regionprops.
    df['sphericity'] = (np.pi**(1/3) * (6 * df['area'])**(2/3)) / df['surface_area']

    return df

In [ ]:
# testing full run
path = r"E:\KristinG\20260625_zebrafish\fin"
#imgs, pixel_spacing = read_imgs(path)

In [3]:
import random

In [4]:
img_files = sorted(glob(r"E:\KristinG\20260625_zebrafish\fin\*.czi"))
rand_files = random.sample(img_files, 2)
pixel_spacing = get_pixel_spacing(BioImage(rand_files[0]))
imgs = [BioImage(f).get_image_data("ZYX", C=1) for f in rand_files]

In [5]:
scale = [0.5,0.2,0.2]
print('normalizing images')
norm_imgs = [normalize_images(i) for i  in tqdm(imgs)]
print('rescaling')
rescaled_imgs = [down_size(i,scale) for i in tqdm(norm_imgs)]
print('running cpsam')
masks = tqdm(run_cpsam(rescaled_imgs))
print('resizing masks')
resized_masks = [up_size_masks(m, i) for m, i in tqdm(zip(masks,norm_imgs))]
print('clearing objects from XY border')
cleared_borders = [clear_xy_borders(m,border=6) for m in tqdm(resized_masks)]
print('getting object measurements')
dfs = [measure_objects(m,pixel_spacing) for m in tqdm(cleared_borders)]

normalizing images


100%|██████████| 2/2 [00:08<00:00,  4.40s/it]


rescaling


100%|██████████| 2/2 [00:02<00:00,  1.29s/it]


running cpsam


...

100%|██████████| 2/2 [00:01<00:00,  1.36it/s]
2it [00:01,  1.36it/s]


clearing objects from XY border


100%|██████████| 2/2 [00:03<00:00,  1.80s/it]


getting object measurements


100%|██████████| 2/2 [03:48<00:00, 114.19s/it]


In [17]:
del(dfs)

In [15]:
print('clearing objects from XY border')
cleared_borders = [clear_xy_borders(m,border=6) for m in tqdm(resized_masks)]
print('getting object measurements')
dfs = [measure_objects(m,pixel_spacing) for m in tqdm(cleared_borders)]

clearing objects from XY border


100%|██████████| 2/2 [00:06<00:00,  3.42s/it]


getting object measurements


  0%|          | 0/2 [00:00<?, ?it/s]

found 46, extracting object regions
Extracting suface areas for objects


 50%|█████     | 1/2 [01:06<01:06, 66.37s/it]

found 119, extracting object regions
Extracting suface areas for objects


100%|██████████| 2/2 [03:27<00:00, 103.82s/it]


In [34]:
dfs = [measure_objects(m,pixel_spacing,n_jobs=10) for m in tqdm(cleared_borders)]

  0%|          | 0/2 [00:00<?, ?it/s]

collecting regions properties
finding objects with ndimage
extracting subregions for 46
Extracting suface areas for objects


 50%|█████     | 1/2 [00:19<00:19, 19.90s/it]

collecting regions properties
finding objects with ndimage
extracting subregions for 119
Extracting suface areas for objects


100%|██████████| 2/2 [00:48<00:00, 24.25s/it]
